# Meme Organizer: 02. Fine-Tuning

**Постановка задачи:**
Плотные модели общего назначения (RuBERT, E5-small) уступают лексическому поиску на специфичном интернет-сленге мемов (*«несет бред»* $\leftrightarrow$ *«в палату быстро»*, *«согласен с другом»* $\leftrightarrow$ *«мой меч с тобой»*).

В данном исследовании представлено дообучение
1. Обучение на 1021 Train-парах через **Multiple Negatives Ranking Loss** с In-Batch Negatives.
2. Применение легковесных **LoRA** адаптеров.
3. Валидация по эпохам на парах из `val_pairs` без заглядывания в тестовый сплит.
4. Экспорт обученных моделей в мобильный формат **ONNX INT8**.
5. Финальная оценка на Test-сплите и сравнение на лидерборде MLflow.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from meme_organizer.dataset.loader import DatasetLoader
from meme_organizer.models.finetuned_dense import FineTunedDenseRetriever
from meme_organizer.pipeline.retrieval_pipeline import RetrievalPipeline

GOLD_DATASET_PATH = Path("../data/gold/gold_dataset.jsonl")
SPLIT_PATH = Path("../data/splits/splits_v1.json")

2026/11/25 15:06:27 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Дообучение легкой модели `rubert-tiny2`

In [2]:
split_info = DatasetLoader.load_split_info(SPLIT_PATH)
print(f"Корпус: {len(split_info['corpus_hashes'])} документов")
print(
    f"Train пары: {len(split_info['train_pairs'])}, Val пары: {len(split_info['val_pairs'])}, Test пары: {len(split_info['test_pairs'])}"
)

Корпус: 318 документов
Train пары: 1021, Val пары: 138, Test пары: 153


## 2. Дообучение мультиязычной модели `multilingual-e5-small`

In [3]:
pipeline = RetrievalPipeline()
rubert_ft = FineTunedDenseRetriever(
    model_name="rubert-tiny2-finetuned",
    base_model_id="cointegrated/rubert-tiny2",
    epochs=50,
    patience=10,
    lr=2e-4,
    lora_r=8,
)

metrics_rubert_ft, run_id_rubert_ft = pipeline.run(
    retriever=rubert_ft,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    train_mode=True,
)

pd.DataFrame(
    [
        {
            "Model": "RuBERT-tiny2 (Fine-tuned)",
            "Recall@1": metrics_rubert_ft.recall_at_1,
            "Recall@5": metrics_rubert_ft.recall_at_5,
            "MRR@5": metrics_rubert_ft.mrr_at_5,
            "NDCG@5": metrics_rubert_ft.ndcg_at_5,
            "MLflow Run ID": run_id_rubert_ft,
        }
    ]
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run rubert-tiny2-finetuned_test at: http://localhost:5000/#/experiments/10/runs/7e3f6bfe4a584599a3f2cfdc52c383f9
🧪 View experiment at: http://localhost:5000/#/experiments/10


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,RuBERT-tiny2 (Fine-tuned),0.4641,0.6078,0.5154,0.5383,7e3f6bfe4a584599a3f2cfdc52c383f9


## 3. Графики обучения по эпохам из MLflow

In [4]:
e5_ft = FineTunedDenseRetriever(
    model_name="multilingual-e5-small-finetuned",
    base_model_id="intfloat/multilingual-e5-small",
    query_prefix="query: ",
    passage_prefix="passage: ",
    epochs=50,
    patience=10,
    lr=1e-4,
    lora_r=16,
)

metrics_e5_ft, run_id_e5_ft = pipeline.run(
    retriever=e5_ft,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
    train_mode=True,
)

pd.DataFrame(
    [
        {
            "Model": "Multilingual-E5-small (Fine-tuned)",
            "Recall@1": metrics_e5_ft.recall_at_1,
            "Recall@5": metrics_e5_ft.recall_at_5,
            "MRR@5": metrics_e5_ft.mrr_at_5,
            "NDCG@5": metrics_e5_ft.ndcg_at_5,
            "MLflow Run ID": run_id_e5_ft,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🏃 View run multilingual-e5-small-finetuned_test at: http://localhost:5000/#/experiments/10/runs/4521aacd070a4195a2cfb1f8cdd739ac
🧪 View experiment at: http://localhost:5000/#/experiments/10


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Multilingual-E5-small (Fine-tuned),0.5686,0.719,0.6337,0.6554,4521aacd070a4195a2cfb1f8cdd739ac


## 4. Итоговый лидерборд экспериментов Домена 1

In [5]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("meme-organizer-retrieval")
runs = client.search_runs(exp.experiment_id, order_by=["attribute.start_time DESC"])

rows = []
for r in runs[:2]:
    m = r.data.metrics
    rows.append(
        {
            "Model": r.data.tags.get("mlflow.runName"),
            "Recall@1": m.get("recall_at_1"),
            "Recall@5": m.get("recall_at_5"),
            "MRR@5": m.get("mrr_at_5"),
            "NDCG@5": m.get("ndcg_at_5"),
            "Train Time (s)": m.get("training_time_sec"),
            "P95 Latency (ms)": m.get("cpu_latency_p95_ms"),
        }
    )
display(pd.DataFrame(rows))

,Model,Recall@1,Recall@5,MRR@5,NDCG@5,Train Time (s),P95 Latency (ms)
0,multilingual-e5-small-finetuned_test,0.5686,0.7190,0.6337,0.6554,1285.079,11.85
1,rubert-tiny2-finetuned_test,0.4641,0.6078,0.5154,0.5383,350.800,3.00
